# 8 · A model told the bitrate and the line

A blind model has to guess how each frame was made. The client knows two things about its stream without reading a
single frame: the **bitrate** it asked for, and which of the host's **lines** sends it: NVENC, which codes with the
loop filter on, or x264 `ultrafast`, which codes without. This notebook trains notebook 7's body told those two, and
the same body blind at three seeds, the ruler for what luck alone moves.

The told model is `models.ToldLuma`: a 1×1 convolution turns the condition into a **shift per channel**, added after
the stem and inside each residual block. It starts at zero, so untrained it is the blind model, and for one stream the
condition is a constant, so it bakes into the biases of a plain `models.LumaUpscaler`: one graph per bitrate and line,
the client's contract unchanged. Notebooks 18–23 tell the model what changes frame by frame instead: the slice QP.

Needs: `00_data_preprocess.ipynb`, `01_upscale_dummy.ipynb` (bicubic's scores) and `04_upscale_strategies.ipynb` (the
blind model at seed 0).

In [1]:
"""Settings."""

import os

import numpy as np
import torch

os.environ["DATASETS_DIR"] = "/mnt/d/datasets"   # where the datasets are: set before helpers is imported

from helpers import models, patches, training
from helpers.training import DEVICE

NOTEBOOK = 8
WIDTH, BLOCKS, STEM = 24, 2, 3       # notebook 7's body
SEEDS = (0, 1, 2)                    # the blind model three times: how far luck alone moves it
TOLD = ("bpp", "deblocked")          # parts of patches.CONDITION_PARTS: the bitrate, and the line's loop filter
TOLD_NAME = "told the bitrate and line"
RUN_DIR = training.CHECKPOINT_DIR / "strategy_runs"   # the blind runs share notebook 4's files

manifest = training.load_manifest()
SCALE = manifest["scale"]
bicubic = training.recorded()[1, "bicubic"]   # notebook 1's scores of the bar
COLUMNS = patches.columns(TOLD)


def build(told=False, seed=0):
    """The blind body, or the told one; the same seed starts both with the same weights."""
    torch.manual_seed(seed)
    if told:
        return models.ToldLuma(len(COLUMNS), WIDTH, BLOCKS, STEM, scale=SCALE).to(DEVICE)
    return models.LumaUpscaler((WIDTH,), BLOCKS, STEM, scale=SCALE).to(DEVICE)

## 1. What the model is told

`patches.conditions` holds every patch's labels: `bpp` is log2 of the stream's bits per pixel against the host's
default and a 1 for no codec; `deblocked` is 1 on the NVENC line. The told model gets them as a second input, the
same for the whole patch.

In [2]:
condition = patches.conditions(training.DATA_DIR, "val", manifest)[:, COLUMNS]
lines = np.load(training.DATA_DIR / manifest["splits"]["val"]["encoder"])
for line, name in enumerate(manifest["encoders"]):
    rows = condition[lines == line]
    print(f"{name:<6} log2 bpp / 0.46 from {rows[:, 0].min():+.2f} to {rows[:, 0].max():+.2f}, "
          f"loop filter {sorted(set(rows[:, 2].tolist()))}")

trim = training.trimmer(manifest)   # cuts the halo off an output: the border is padding, not picture
x = torch.rand(8, 3, manifest["input_lr"], manifest["input_lr"], device=DEVICE)
c = torch.randn(8, len(COLUMNS), 1, 1, device=DEVICE)
blind, told = build(), build(told=True)
with torch.no_grad(), training.full_precision():
    for name, out in (("blind", blind(x)), ("told", told(x, c))):
        gap = (trim(out) - trim(models.bicubic(x, SCALE))).abs().max().item()
        assert gap < 1e-5, f"untrained {name} is not bicubic ({gap:.1e})"
    # random weights, then one condition baked into the plain model: the same output
    for weight in told.parameters():
        if weight.requires_grad:
            weight.normal_(0, 0.1)
    gap = (told(x, c[:1].expand(8, -1, -1, -1)) - told.baked(c[:1].flatten())(x)).abs().max().item()
assert gap < 1e-4, f"baking changed the output ({gap:.1e})"
print(f"blind: {models.count_parameters(blind, True):,} learned parameters, told: "
      f"{models.count_parameters(told, True):,}; baking a condition is exact ({gap:.1e})")

x264   log2 bpp / 0.46 from -5.58 to +2.33, loop filter [0.0]
nvenc  log2 bpp / 0.46 from -5.58 to +2.40, loop filter [1.0]
blind: 13,156 learned parameters, told: 13,372; baking a condition is exact (9.5e-07)


## 2. Train

`training.fit` for each, the shared recipe. The told model and the blind one at seed 0 start from the same weights and
see the same batches in the same order, so the gap between them is what the condition buys; the three blind seeds say
how large luck is. The blind run at seed 0 is notebook 4's `luma 24 + 2 blocks`, loaded from its file.

In [ ]:
runs = {}
for seed in SEEDS:
    model = build(seed=seed)
    history, best = training.fit(model, RUN_DIR / f"luma_24_2_blocks_seed{seed}.pt", manifest, seed=seed, verbose=0)
    runs[f"blind, seed {seed}"] = model.eval(), history
model = build(told=True)
history, best = training.fit(model, training.CHECKPOINT_DIR / "told_runs" / "told_bpp_deblocked_seed0.pt", manifest,
                             told=TOLD, verbose=0)
runs[TOLD_NAME] = model.eval(), history
for name, (model, history) in runs.items():
    print(f"{name:<28} {len(history):3} epochs")

training.plot_history({name: history for name, (_, history) in runs.items()}, training.select_bicubic(manifest),
                      key="ssim", loss_name="SSIM-Y + L1 loss")

## 3. Scores

Every run on the validation split; the told model is given each patch's own condition. Then the told model against
the **mean** of the blind seeds: a gap within their spread is luck.

In [ ]:
scores = {}
for name, (model, _) in runs.items():
    if name == TOLD_NAME:
        scores[name] = training.val_scores(lambda lr, condition: model(lr, condition[:, COLUMNS]), manifest)
    else:
        scores[name] = training.val_scores(lambda lr, condition: model(lr), manifest)
for name in ("blind, seed 1", "blind, seed 2", TOLD_NAME):   # notebook 4 recorded seed 0
    training.record(NOTEBOOK, f"luma 24 + 2 blocks, seed {name[-1]}" if name.startswith("blind") else name,
                    scores[name])

training.show(training.compare({"bicubic": bicubic, **scores}, manifest, base=bicubic))

blind_runs = [scores[f"blind, seed {seed}"] for seed in SEEDS]
blind_mean = {key: np.mean([s[key] for s in blind_runs], axis=0) for key in ("ssim", "y")}
spread = np.ptp([s["ssim"].mean() for s in blind_runs])
print(f"\nthe blind model at {len(SEEDS)} seeds: {spread:.4f} SSIM-Y apart")
table = training.compare({TOLD_NAME: scores[TOLD_NAME]}, manifest, base=blind_mean, base_name="blind")
table["verdict"] = ["luck" if abs(gap) <= spread else "better" if gap > 0 else "worse" for gap in table["over blind"]]
training.show(table)
print()
training.show(training.gain_by_qp({"blind mean": blind_mean, TOLD_NAME: scores[TOLD_NAME]}, manifest, base=bicubic))

## 4. What it says

*(written after the run)*